In [ ]:
import asyncio
import base64
import hashlib
import json
from pathlib import Path

from IPython.display import display

import xyg
from test_graphforge import _base
from xyg import _native
from xyg._graph_session import GraphSession, GraphSessionError, GraphSessionStatus
from xyg._graphforge_container import DTYPE as D
from xyg._graphforge_container import encode_container

composition = xyg.compose_graphforge(base=_base("property"))
capacity_owners = []
for instance in range(1, 258):
    opening = encode_container(
        "XYGI",
        [
            ("header.u64", 0, D.U64, [instance, 0, 1]),
            ("header.u32", 0, D.U32, [0, 19, 0, 0, 0, 0, 0, 0, 0]),
            ("document.bytes", 0, D.BYTES, composition.bytes),
        ],
    )
    try:
        capacity_owners.append(
            GraphSession(composition.bytes, opening, instance_id=instance, group_id=0)
        )
    except GraphSessionError as error:
        assert error.status == GraphSessionStatus.CAPACITY
        break
else:
    raise AssertionError("native owner table did not enforce capacity")
assert capacity_owners
selections, brushes, other_selections = [], [], []
report_sequence = 0


def selected(value):
    selections.append(
        {
            "identities": value.graphforge,
            "per_trace": {str(k): v.tolist() for k, v in value.per_trace.items()},
            "rows": value.rows(),
        }
    )


chart = xyg.graphforge_chart(
    composition,
    width=900,
    height=420,
    on_select=selected,
    on_brush=brushes.append,
    title="Native graph regions",
)
widget = chart.widget()
other = xyg.graphforge_chart(
    composition,
    width=900,
    height=420,
    on_select=lambda value: other_selections.append(value.rows()),
    title="Isolated native graph",
).widget()
display(widget)
display(other)

In [ ]:
report_sequence += 1
figure = widget._figure
meta = figure._graph_meta[0]
identity = figure._notebook_graph_identity
edge_identity = figure._graph_edge_identity[meta["edge_trace"]]
fixture = {
    "composition": base64.b64encode(composition.bytes).decode("ascii"),
    "x": identity._positions[0].tolist(),
    "y": identity._positions[1].tolist(),
    "node_trace": meta["node_trace"],
    "edge_trace": meta["edge_trace"],
    "member_of": list(map(int, meta["member_of"])),
    "edge_rows": [
        edge_identity.members_of(int(i)).tolist() for i in edge_identity.render_edge_index
    ],
    "node_ids": list(composition.nodes.uuid),
    "edge_ids": list(composition.edges.uuid),
    "edge_source": composition.edges.source.tolist(),
    "edge_target": composition.edges.target.tolist(),
}
if report_sequence == 1:
    lo, hi = min(fixture["y"][:4]), max(fixture["y"][:4])
    widget.set_view({"y": [lo - 0.4, hi + 0.4]}, animate=False)
    widget.select(rows={meta["node_trace"]: [0], meta["edge_trace"]: [0]})
print(
    "XYG graph region report",
    json.dumps(
        {
            "report_sequence": report_sequence,
            "fixture": fixture,
            "capacity_owners": len(capacity_owners),
            "selections": selections,
            "brushes": brushes,
            "other_selections": other_selections,
            "state": widget.view_state(),
            "native_sha256": hashlib.sha256(Path(_native._lib._name).read_bytes()).hexdigest(),
        }
    ),
)

In [ ]:
capacity_owners.pop().close()
widget.clear_selection()
print("XYG graph region capacity released", len(capacity_owners))

In [ ]:
owners = [
    f._notebook_graph_identity.session
    for w in (widget, other)
    for f in w._figures.values()
    if f._notebook_graph_identity.session is not None
]
widget.close()
other.close()
for owner in capacity_owners:
    owner.close()
await asyncio.sleep(0)
assert all(owner._handle is None for owner in [*owners, *capacity_owners])
print("XYG graph region cleanup", json.dumps({"owners": len(owners), "released": True}))